    Importing packages.

In [1]:
import torch
from torch import nn
from torch.nn import functional as F
from torch.nn.utils import parametrize as param
from torch.linalg import multi_dot as md
import os
import pickle
from torchvision import datasets, transforms
from torch.autograd import Variable
from torch.utils.data import Dataset, DataLoader, Subset
from torch._utils import _accumulate
from tqdm import trange, tqdm
import numpy as np
import numpy.random as npr
import pandas as pd
import matplotlib.pyplot as plt

    Defining the model.

In [2]:
def random_lambda(n:int, k:int, g:float, gb:float, gc:float):
    """Function which generates a random vector for SigmaA1 when k>1,
        subject to the k-contraction constraint - refer to Algorithm 1.
        args:
            n: dimension of the vector.
            k: integer between 2 and n which controls the dimension of the k-compound system.
            g: positive real value describing the upper bound on the slope of the nonlinearity.
           gb: positive real value describing the upper bound on the sum of the k-largest singular values of the B matrix.
           gc: positive real value describing the upper bound on the sum of the k-largest singular values of the C matrix.
        returns:
                square n x n matrix with the eigenvalues of the symmetric component of A (which satisfy Theorem 2) along the main diagonal.
    """
    if k<2:
        raise ValueError(f"This function should only be used for k>1!")

    sk = -2*g*gb*gc/k + 1 # +1 is just to satisfy while loop conditions
    sk_1 = -2*g*gb*gc/(k-1) - 1 # -1 is just to satisfy while loop conditions
    r1 = -2*g*gb*gc/((k-1)**2)
    r2 = 2*g*gb*gc/(k**2)
    r_lambda = torch.zeros(n)

    while sk >= -2*g*gb*gc/k or sk_1 < -2*g*gb*gc/(k-1):
        r_lambda[0:k] = torch.sort((r2 - r1)*torch.rand(k) + r1*torch.ones(k),descending=True).values # ~ U(r1,r2)
        sk_1 = torch.sum(r_lambda[0:k-1])
        sk = sk_1 + r_lambda[k-1]

    r_lambda[k:n] = r_lambda[k-1]*torch.ones(n-k) - torch.rand(n-k)

    return torch.diag(r_lambda)

def get_init(output_size:int, input_size:int, init:str, std=0.1):
    """Returns a random initialisation following using the scheme init.
        Initialisation is only for weight matrices, bias' are set to zero.
        args:
            output_size = dimension of matrix output.
            input_size = dimension of matrix input.
            std: standard deviation of normal distribution.
           init: the initialisation scheme to use.
        returns: tensor of shape [output_size, input_size] sampled from specified normal distribution.
        """
    if init=='He':
        return torch.normal(0,2*std/input_size,(output_size,input_size))
    elif init=='Xavier':
        return torch.normal(0,std/input_size,(output_size,input_size)) # used in LipschitzRNN
    elif init=='RNNsofRNNs':
        return torch.normal(0,std/np.sqrt(input_size), (output_size,input_size))
    elif init=='He+RNNsofRNNs':
        return torch.normal(0,2*std/np.sqrt(input_size), (output_size,input_size)) # made up
    else:
        raise ValueError(f"Unspecified initialisation.")



In [3]:
class LurieNet(nn.Module):
    """Class for constructing the Lurie network.
        args:
            input_size: dimension of the Lurie network input.
           output_size: dimension of the Lurie network output.
                     n: dimension of the Lurie network state (x).
                     m: dimension of the argument to the nonlinearity (y).
                     k: integer in [1,n] which controls the dimension of the k-compound system. This class only works for k=1.
                     g: positive real value describing the upper bound on the slope of the nonlinearity.
                    gb: positive real value describing the upper bound on the sum of the k-largest singular values of the B matrix.
                    gc: positive real value describing the upper bound on the sum of the k-largest singular values of the C matrix.
                   ga1: positive real value for lower bounding the eigenvalues of the symmetric component of A.
                   ga2: positive real value for upper bounding the magnitude of the skew-symmetric component of A.
                 delta: positive real value for discretizing continuous time dynamics.
                v1_ind: boolean variable for indicating if input v1 is present in model.
                v2_ind: boolean variable for indicating if input v2 is present in model.
                  init: string containing the specified weight initialisation.
optional args:
                     A: nxn tensor for passing in a A matrix when it is not being learnt.
                     B: nxm tensor for passing in a B matrix when it is not being learnt.
                     C: mxn tensor for passing in a C matrix when it is not being learnt.
    functions:
            forward: one step computes one step ahead of the Lurie network.
    """

    def __init__(self, input_size, output_size, n, m, k, g, gb, gc, ga1, ga2, delta, v1_ind, v2_ind, init, A=None, B=None, C=None):
        super().__init__()
        self.n = n
        self.m = m
        self.r = min(n,m)
        self.k = k
        self.g = g
        self.gb = gb
        self.gc = gc
        self.ga1 = ga1
        self.ga2 = ga2
        self.delta = delta
        self.v1_ind = v1_ind
        self.v2_ind = v2_ind
        self.A = A
        self.B = B
        self.C = C

        # Instantiate input layer(s) and initialise bias' to zero.
        if v1_ind == True:
            self.V1_in = nn.Linear(input_size, n)
            self.V1_in.bias.data = torch.zeros(self.V1_in.bias.data.shape[0])
        if v2_ind == True:
            self.V2_in = nn.Linear(input_size, m)
            self.V2_in.bias.data = torch.zeros(self.V2_in.bias.data.shape[0])
        if v1_ind == False and v2_ind == False:
            raise ValueError(f"No inputs!.")

        # Instantiate output layer and initialise bias to zero.
        self.out = nn.Linear(n, output_size)
        self.out.bias.data = torch.zeros(output_size)

        # defining number of parameters in skew symmetric matrices and calculating lower triangular indices for nxn and mxm matrices
        skew_n = int(n*(n-1)/2)
        skew_m = int(m*(m-1)/2)
        self.tril_ind_n = torch.tril_indices(n, n, offset=-1) # lower triangular indices
        self.tril_ind_m = torch.tril_indices(m, m, offset=-1) # lower triangular indices

        if C==None:
            # Components of C matrix - UC, VC just need to be skew symmetric, SC just needs to be diagonal with r non-zero elements.
            self.UC = nn.Parameter(get_init(1,skew_m,init))
            self.SC = nn.Parameter(get_init(1,self.r,init))
            self.VC = nn.Parameter(get_init(1,skew_n,init))

        if B==None:
            # Components of B matrix - UB, VB just need to be skew symmetric, SB just needs to be diagonal with r non-zero elements.
            self.UB = nn.Parameter(get_init(1,skew_n,init))
            self.SB = nn.Parameter(get_init(1,self.r,init))
            self.VB = nn.Parameter(get_init(1,skew_m,init))

        if A==None:
            # Components of A matrix  - UA1, UA2 just need to be skew symmetric, SA1 just need to be diagonal with n non-zero elements, and
            # SA2 has a special skew-symmetric form with n-1 non-zero elements.
            self.UA1 = nn.Parameter(get_init(1,skew_n,init))
            self.UA2 = nn.Parameter(get_init(1,skew_n,init))
            self.SA2 = nn.Parameter(get_init(1,n-1,init))
            if k==1:
                self.SA1 = nn.Parameter(get_init(1,n,init))
            elif k>1 and k<=n:
                self.SA1 = random_lambda(n, k, g, gb, gc)
            else:
                raise ValueError(f"Invalid choice of k.")

    def forward(self, inputs):
        """Constructs the LurieNet layer subject to the k-contraction parameterization. Then passes pixels into the network one-by-one, but
           in parrallel for all images in the batch. Hence, vectors x, y, v1, v2 are expressed as matrices below, instead of vectors as described in the paper.
            args:
                self: see above.
                inputs: tensor of shape [batch_size, pixels/image, rgb=input_size]
         returns: tensor of shape [batch size, output size] containing "likelihoods" of each class (prediction = highest likelihood).
        """

        X = torch.zeros(inputs.shape[0], self.n) # shape [batch_size, n]

        if self.C==None:
            # Construct C matrix
            UC_temp = torch.zeros(self.m,self.m)
            UC_temp[self.tril_ind_m[0], self.tril_ind_m[1]] = self.UC # represents self.UC vector as a lower triangular matrix.
            UC = torch.matrix_exp(UC_temp - torch.transpose(UC_temp,0,1)) # Orthogonal parametrization of self.UC
            VC_temp = torch.zeros(self.n,self.n)
            VC_temp[self.tril_ind_n[0], self.tril_ind_n[1]] = self.VC # represents self.VC vector as a lower triangular matrix.
            VC = torch.matrix_exp(VC_temp - torch.transpose(VC_temp,0,1)) # Orthogonal parametrization of self.VC
            Cmask = torch.zeros(self.m, self.n)
            Cmask[0:self.r,0:self.r] = torch.eye(self.r) # Mask to ensure SC has 0 elements for all but r elements on the main diagonal.
            SC_temp = torch.zeros(self.m, self.n)
            SC_temp[[i for i in range(self.r)], [i for i in range(self.r)]] = self.SC # embeds self.SC in an mxn matrix along first r elements on main diagonal.
            SC = (self.gc/self.k)*Cmask*torch.exp(-SC_temp**2) # exp(0)=1, so need mask to set all elements to zero other than first r along main diagonal.
            C = md([UC, SC, torch.transpose(VC,0,1)])
        else:
            C = self.C # Pass in C matrix

        if self.B==None:
            # Construct B matrix
            UB_temp = torch.zeros(self.n,self.n)
            UB_temp[self.tril_ind_n[0], self.tril_ind_n[1]] = self.UB # represents self.UB vector as a lower triangular matrix.
            UB = torch.matrix_exp(UB_temp - torch.transpose(UB_temp,0,1)) # Orthogonal parametrization of self.UB
            VB_temp = torch.zeros(self.m,self.m)
            VB_temp[self.tril_ind_m[0], self.tril_ind_m[1]] = self.VB # represents self.VB vector as a lower triangular matrix.
            VB = torch.matrix_exp(VB_temp - torch.transpose(VB_temp,0,1)) # Orthogonal parametrization of self.VB
            Bmask = torch.zeros(self.n, self.m)
            Bmask[0:self.r,0:self.r] = torch.eye(self.r) # Mask to ensure SB has 0 elements for all but r elements on the main diagonal.
            SB_temp = torch.zeros(self.n, self.m)
            SB_temp[[i for i in range(self.r)], [i for i in range(self.r)]] = self.SB # embeds self.SB in an nxm matrix along first r elements on main diagonal.
            SB = (self.gb/self.k)*Bmask*torch.exp(-SB_temp**2) # exp(0)=1, so need mask to set all elements to zero other than first r along main diagonal.
            B = md([UB, SB, torch.transpose(VB,0,1)])
        else:
            B = self.B # Pass in B matrix

        if self.A==None:
            # Construct A matrix
            UA1_temp = torch.zeros(self.n,self.n)
            UA1_temp[self.tril_ind_n[0], self.tril_ind_n[1]] = self.UA1 # represents self.UA1 vector as a lower triangular matrix.
            UA1 = torch.matrix_exp(UA1_temp - torch.transpose(UA1_temp,0,1)) # Orthogonal parametrization of self.UA1
            UA2_temp = torch.zeros(self.n,self.n)
            UA2_temp[self.tril_ind_n[0], self.tril_ind_n[1]] = self.UA2 # represents self.UA2 vector as a lower triangular matrix.
            UA2 = torch.matrix_exp(UA2_temp - torch.transpose(UA2_temp,0,1)) # Orthogonal parametrization of self.UA2
            A2mask = torch.diag(torch.ones(self.n-1),diagonal=1) # Mask to ensure SA2 has 0 elements for all but n-1 elements above the main diagonal.
            SA2_temp = torch.zeros(self.n,self.n)
            SA2_ind_x = []
            SA2_ind_y = []
            for i in range(self.n): # row
                for j in range(self.n): # col
                    if i-j ==-1:
                        SA2_ind_x.append(i)
                        SA2_ind_y.append(j)
            SA2_temp[SA2_ind_x,SA2_ind_y] = self.SA2 # embeds self.SA2 vector above the main diagonal.
            SA2 = self.ga2*A2mask*torch.exp(-SA2_temp**2) - torch.transpose(self.ga2*A2mask*torch.exp(-SA2_temp**2),0,1) # exp(0)=1, so need mask to set all elements to zero other than those above the main diagonal.
            if self.k == 1:
                A1mask = torch.eye(self.n) # Mask to ensure SA1 has 0 elements for all but n elements along the main diagonal.
                A1_temp = torch.zeros(self.n,self.n)
                A1_temp[[i for i in range(self.n)], [i for i in range(self.n)]] = self.SA1 # embeds self.SA1 vector along the main diagonal.
                SA1 = -2*self.g*self.gb*self.gc*torch.eye(self.n) - self.ga1*A1mask*F.tanh(A1_temp**2) - (10e-5)*torch.eye(self.n)
            elif self.k>1 and self.k<=self.n:
                SA1 = self.SA1 # determined by r_lambda() in initialisation
            else:
                raise ValueError(f"Invalid choice of k.")
            A = 0.5*md([UA1, SA1, torch.transpose(UA1,0,1)]) + 0.5*md([UA2, SA2, torch.transpose(UA2,0,1)])
        else:
            A = self.A # Pass in A matrix

        # Loop through pixels in an image. inputs[:,i,:] has shape [batch_size, input_size] and inputs has shape [batch_size, pixels, input_size]
        for i in range(inputs.shape[1]):
            if self.v1_ind==True and self.v2_ind==False:
                V1 = self.V1_in(inputs[:,i,:]) # V1 has shape [batch size, n]
                Y = torch.matmul(X, torch.transpose(C,0,1)) # Y has shape [batch size, m]
                X = X + self.delta*(torch.matmul(X, torch.transpose(A,0,1)) + torch.matmul(F.relu(Y), torch.transpose(B,0,1)) + V1) # X has shape [batch size, n]

            elif self.v1_ind==False and self.v2_ind==True:
                V2 = self.V2_in(inputs[:,i,:])
                Y = torch.matmul(X, torch.transpose(C,0,1)) + V2 # Y has shape [batch size, m]
                X = X + self.delta*(torch.matmul(X, torch.transpose(A,0,1)) + torch.matmul(F.relu(Y), torch.transpose(B,0,1))) # X has shape [batch size, n]

            elif self.v1_ind==True and self.v2_ind==True:
                V1 = self.V1_in(inputs[:,i,:])
                V2 = self.V2_in(inputs[:,i,:]) # V2 has shape [batch size, m]
                Y = torch.matmul(X, torch.transpose(C,0,1)) + V2 # Y has shape [batch size, m]
                X = X + self.delta*(torch.matmul(X, torch.transpose(A,0,1)) + torch.matmul(F.relu(Y), torch.transpose(B,0,1)) + V1) # X has shape [batch size, n]

            else:
                raise ValueError(f"LurieNet has no inputs!")

        # dictionary of component matrices - for testing purposes!
        # dict = {'UC':UC, 'SC':SC, 'VC':VC, 'UB':UB, 'SB':SB, 'VB':VB, 'UA1':UA1, 'SA1':SA1, 'UA2':UA2, 'SA2':SA2}

        return self.out(X)

    Loading, preproccessing and splitting data.

In [4]:
def add_channels(X):
    """ Helper function when permutting input data for psmnist task. Dependent on the shape of the permutted dataset,
        the dataset is reshaped.
        args:
            X: permutted dataset
     returns: reshaped permutted dataset
    """
    if len(X.shape) == 2:
        return X.reshape(X.shape[0], 1, X.shape[1], 1)
    elif len(X.shape) == 3:
        return X.reshape(X.shape[0], 1, X.shape[1], X.shape[2])
    else:
        return "Dataset has unexpected dimensions!"

In [5]:
def getData(name:str, dir:str, device:torch.device, r_seed:int, train_bs=64, test_bs=1024):
    """ Function for loading, preprocessing and splitting data for a specified task. Note that LurieNet requires
        tensor containing inputs of shape (N,input_size) where N indicates the batch size.
        args:
            name: String specifying the task. Accepted arguments are scifar10, smnist, psmnist.
             dir: String specifying the directory where the dataset should be saved.
          device: Parameter indicating the type of hardware in use.
          r_seed: Random seed for random generator.
        train_bs: Integer stating the batch size during training (default 64).
         test_bs: Integer stating the batch size during testing (default 1024).
     returns: train, validation and test set dataloaders.
    """

    # Same transformations as RNNs of RNNs (Kozachkov, et al. 2022).
    if name == 'scifar10':
        transform_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
    ])

        transform_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
    ])
        # Returns (image, target) tuple of full train set and test set, where target is index of the target class.
        # Note that if dataset are already downloaded, they are not downloaded again.
        training_data = datasets.CIFAR10(root=dir, train=True, download=True, transform=transform_train)
        testing_data = datasets.CIFAR10(root=dir, train=False, download=True, transform=transform_test)

        # Same settings as RNNs of RNNs (Kozachkov, et al. 2022).
        offset = 2000
        rng = np.random.RandomState(r_seed)
        R = rng.permutation(len(training_data))
        lengths = (len(training_data) - offset, offset)

        # splits full dataset into train and validation set. Overwrites training_data to save memory.
        training_data, val_data = [Subset(training_data, R[offset - length:offset]) for offset, length in zip(_accumulate(lengths), lengths)]

        # Specifying generator for reproducbiiltiy.
        generator = torch.Generator(device=device)
        generator.manual_seed(r_seed+1)

        # Creating dataloaders - iterables that pass samples in minibatches and reshuffle data every epoch (if shuffle=True).
        train_loader = torch.utils.data.DataLoader(training_data, batch_size=train_bs, shuffle=True, generator=generator)
        val_loader = torch.utils.data.DataLoader(val_data, batch_size=test_bs, shuffle=False, generator=generator)
        test_loader = torch.utils.data.DataLoader(testing_data, batch_size=test_bs, shuffle=False, generator=generator)

    if name == 'smnist':

        training_data = datasets.MNIST(dir, train=True, download=True, transform=transforms.Compose([transforms.ToTensor(),]))
        testing_data = datasets.MNIST(dir, train=False, download=True, transform=transforms.Compose([transforms.ToTensor(),]))

        # Same settings as RNNs of RNNs (Kozachkov, et al. 2022).
        offset = 2000
        rng = np.random.RandomState(r_seed+2)
        R = rng.permutation(len(training_data))
        lengths = (len(training_data) - offset, offset)

        # splits full dataset into train and validation set. Overwrites training_data to save memory.
        training_data, val_data = [Subset(training_data, R[offset - length:offset]) for offset, length in zip(_accumulate(lengths), lengths)]

        # Specifying generator for reproducbiiltiy.
        generator = torch.Generator(device=device)
        generator.manual_seed(r_seed+3)

        train_loader = torch.utils.data.DataLoader(training_data, batch_size=train_bs, shuffle=True, generator=generator)
        val_loader = torch.utils.data.DataLoader(val_data, batch_size=test_bs, shuffle=False, generator=generator)
        test_loader = torch.utils.data.DataLoader(testing_data,batch_size=test_bs, shuffle=False, generator=generator)

    if name == 'psmnist':

        training_data = datasets.MNIST(root=dir, train=True, download=True, transform=transforms.Compose([transforms.ToTensor(),]))
        testing_data = datasets.MNIST(root=dir, train=False, download=True, transform=transforms.Compose([transforms.ToTensor(),]))

        # splitting data into features (x) and targets (y)
        x_train = training_data.train_data
        y_train = training_data.targets
        x_test = testing_data.test_data
        y_test = testing_data.targets

        # permutting input data - same settings as RNNs of RNNs (Kozachkov, et al. 2022).
        torch.manual_seed(r_seed+4)
        perm = torch.randperm(x_train.shape[1]*x_train.shape[2])
        x_train_permuted = x_train.reshape(x_train.shape[0],-1)
        x_train_permuted = x_train_permuted[:, perm]
        x_train_permuted = x_train_permuted.reshape(x_train.shape[0], 28, 28)
        x_test_permuted = x_test.reshape(x_test.shape[0],-1)
        x_test_permuted = x_test_permuted[:, perm]
        x_test_permuted = x_test_permuted.reshape(x_test.shape[0], 28, 28)
        x_train_permuted = add_channels(x_train_permuted)
        x_test_permuted = add_channels(x_test_permuted)

        # Forming the psmnist datasets. Overwrites training_data and testing_data to save memory.
        training_data = torch.utils.data.TensorDataset(x_train_permuted.float(), y_train)
        testing_data = torch.utils.data.TensorDataset(x_test_permuted.float(), y_test)

        # Same settings as RNNs of RNNs (Kozachkov, et al. 2022).
        offset = 2000
        rng = np.random.RandomState(r_seed+5)
        R = rng.permutation(len(training_data))
        lengths = (len(training_data) - offset, offset)

        # splits full dataset into train and validation set. Overwrites training_data to save memory.
        training_data, val_data = [Subset(training_data, R[offset - length:offset]) for offset, length in zip(_accumulate(lengths), lengths)]

        # Specifying generator for reproducbiiltiy.
        generator = torch.Generator(device=device)
        generator.manual_seed(r_seed+6)

        # creating dataloaders
        train_loader = torch.utils.data.DataLoader(training_data, batch_size=train_bs, shuffle=True, generator=generator)
        val_loader = torch.utils.data.DataLoader(val_data, batch_size=test_bs, shuffle=False, generator=generator)
        test_loader = torch.utils.data.DataLoader(testing_data,batch_size=test_bs, shuffle=False, generator=generator)

    return train_loader, val_loader, test_loader


    Training loop.

In [6]:
def lr_scheduler(epoch:int, optimizer, decay_eff:float, decayEpoch):
    """Decay learning rate by a factor of decay_eff every epoch is equal to an element in decayEpoch.
        args:
            epoch: current epoch to compare against.
            optimizer: the optimizer for which we wish to update the learning rate.
            decay_eff: the factor to multiply the current learning rate by.
            decayEpoch: list of epoch's which the learning rate should be decreased at.
        returns: optimizer with updated learning rate.
        """
    if epoch in decayEpoch:
        for param_group in optimizer.param_groups:
            param_group['lr'] *= decay_eff
        print('New learning rate is: ', param_group['lr'])

    return optimizer

In [7]:
def train(task:str, model, optimizer, criterion, train_loader, test_loader, max_epoch:int, decay_eff:float, decay_epochs:int, device, output_data:str, output_model:str, min_epoch=0,
          test_accs=[], train_losses=[], epochs_list=[], SA1_max=[], SA1_min=[], SA2_max=[], SA2_min=[], SB_max=[], SB_min=[], SC_max=[], SC_min=[]):
    ''' Main loop for training LurieNet on sCIFAR-10, sMNIST, psMNIST datasets.
        args:
            task: string declaring the task to be trained on. Accepts "scifar10", "smnist", "psmnist".
            model: instance of the LurieNet to be trained.
            optimizer: the chosen optimizer.
            criterion: chosen loss function.
            train_loader: data loader with training samples.
            test_loader: data loader with test samples.
            max_epoch: number of epochs.
            decay_eff: the scalar to multiply the learning rate by at the specified epochs.
            decay_epochs: list containing the epochs to perform a learning rate cut on.
            device: hardware which the code is being run on.
            output_data: path to the logged data.
            output_model: path to the stored models.
optional args:
            min_epoch: epoch to begin training from (default 0). Useful for resuming training after interruption.
            test_accs: list to store test accuracy after each epoch (default empty). Useful for passing in previously recorded list before interruption.
         train_losses: list to store train losses after each epoch (default empty). Useful for passing in previously recorded list before interruption.
          epochs_list: list of epochs which have already passed (counter starts from 1). Useful for passing in previously recorded list before interruption.
              SA1_max: list of max eigenvalues in SA1 over time.
              SA1_min: list of min eigenvalues in SA1 over time.
              SA2_max: list of max eigenvalues in SA2 over time.
              SA2_min: list of min eigenvalues in SA2 over time.
              SB_max: list of max eigenvalues in SB over time.
              SB_min: list of min eigenvalues in SB over time.
              SC_max: list of max eigenvalues in SC over time.
              SC_min: list of min eigenvalues in SC over time.
     returns:
            model: optimized model.
            optimizer: optimizer with up to date parameter choices.
            test_accs: history of test accuracies for each epoch.
            train_losses: history of mean (across batches) loss for each epoch.
    '''

    if task != "scifar10" and task != "smnist" and task != "psmnist":
        raise ValueError('This task cannot be run!')

    optim_params = list(model.parameters())

    for epoch in trange(min_epoch, max_epoch): # trange displays progress meter of loops

        model.train()
        loss_epoch = []

        for i, (inputs, targets) in enumerate(train_loader):

            inputs, targets = inputs.to(device), targets.to(device) # moves data to gpu if available

            # CIFAR-10 images are rgb = 3 and 32x32 = 1024 pixels. MNIST images are rgb = 1 and 28x28 = 784 pixels.
            # Input will be of shape [batch size, rgb, width, height].
            # Want rgb to be the last variable so we can present image pixel by pixel.
            if task == "scifar10":
                inputs = inputs.view(-1, 3, int(1024)).permute(0,2,1) # [batch size, pixels, rgb] = [batch size, 1024, 3]
            else:
                inputs = inputs.view(-1, int(784)).unsqueeze(dim = 2) # [batch size, pixels, rgb] = [batch size, 784, 1]

            # forward + backward + optimize
            outputs = model(inputs) # outputs has shape [batch size, output_size]
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            loss_epoch.append(loss.item())

        # track loss over time, and have it reflect the mean loss over an epoch so it is more reflective of training trends than last batch loss.
        mean_loss = np.mean(loss_epoch)
        train_losses.append(mean_loss)

        # track eigen/singular values over time
        if model.A == None:
          SA1_max.append(torch.min(torch.abs(model.SA1)).to('cpu').detach().numpy()) # smallest mag of SA1 corresponds to SA1_max when passed through parametrization
          SA1_min.append(torch.max(torch.abs(model.SA1)).to('cpu').detach().numpy()) # largest mag of SA1 corresponds to SA1_min when passed through parametrization
          SA2_max.append(torch.min(torch.abs(model.SA2)).to('cpu').detach().numpy()) # smallest mag of SA2 corresponds to SA2_max when passed through parametrization
          SA2_min.append(torch.max(torch.abs(model.SA2)).to('cpu').detach().numpy()) # largest mag of SA2 corresponds to SA2_min when passed through parametrization
        if model.B == None:
          SB_max.append(torch.min(torch.abs(model.SB)).to('cpu').detach().numpy()) # smallest mag of SB corresponds to SB_max when passed through parametrization
          SB_min.append(torch.max(torch.abs(model.SB)).to('cpu').detach().numpy()) # largest mag of SB corresponds to SB_min when passed through parametrization
        if model.C == None:
          SC_max.append(torch.min(torch.abs(model.SC)).to('cpu').detach().numpy()) # smallest mag of SC corresponds to SC_max when passed through parametrization
          SC_min.append(torch.max(torch.abs(model.SC)).to('cpu').detach().numpy()) # largest mag of SC corresponds to SC_min when passed through parametrization

        # Note: when logging data epoch 1 represents LurieNet after first pass over the data and epoch zero represents initial model.
        epoch_log = epoch + 1
        print('Epoch {}, mean batch loss {}'.format(epoch_log, mean_loss))

        # Update learning rate when specified.
        optimizer = lr_scheduler(epoch, optimizer, decay_eff, decay_epochs)

        # Testing model.
        model.eval()
        with torch.no_grad():
            total = 0
            correct = 0
            for (inputs, targets) in test_loader:
                inputs, targets = inputs.to(device), targets.to(device) # moves data to gpu if available.

                if task == "scifar10":
                    inputs = inputs.view(-1, 3, int(1024)).permute(0,2,1)
                else:
                    inputs = inputs.view(-1, int(784)).unsqueeze(dim = 2)

                outputs = model(inputs)

                _, predicted = torch.max(outputs, 1) # predicted: index of the class with the highest value for each image in batch.
                total += targets.shape[0] # updates total with the number images in the batch.
                correct += (predicted == targets).sum().item() # updates correct with the number of correct predictions from the batch.

            print('Accuracy of the network on the %d test images: %d %%' % (total, 100 * correct / total))
            test_accs.append((100.0 * float(correct) / float(total))) # Used floats to get exact accuracy.

        epochs_list.append(epoch_log)

        # save model for every epoch as storage space required is quite small, can have training disruptions with colab
        model_path = os.path.join(output_model,"epoch" + str(epoch_log) + ".pt")
        torch.save(model.state_dict(), model_path)

        # save stats so far, will overwrite every time as rows are added to dataframe.
        stats_path = os.path.join(output_data, "stats.csv")
        cur_stats = pd.DataFrame()
        cur_stats["epoch"] = epochs_list
        cur_stats["loss"] = train_losses # loss over time: indicating mean loss over the batches so it is more reflective of training trends than last batch loss.
        cur_stats["test_acc"] = test_accs # test acc after each epoch.

        # eigen/singular values over time.
        if model.A == None:
          cur_stats["SA1_max"] = SA1_max
          cur_stats["SA1_min"] = SA1_min
          cur_stats["SA2_max"] = SA2_max
          cur_stats["SA2_min"] = SA2_min
        if model.B == None:
          cur_stats["SB_max"] = SB_max
          cur_stats["SB_min"] = SB_min
        if model.C == None:
          cur_stats["SC_max"] = SC_max
          cur_stats["SC_min"] = SC_min

        cur_stats.to_csv(stats_path, index=False)

    return model, optimizer, test_accs, train_losses


    Main loops:

        1. For setting up a new experiment: defining hyperparameters for task, model, training, logging and calling training loop.
        2. For resuming an experiment.

In [9]:
'''
1. Colab: new training run - use this cell when initialising a new training run on Colab.
Use cell below to resume a training run.
'''

# ### settings for running training loop using a specified Google Drive account
from google.colab import drive
drive.mount('/content/gdrive')
gdrive_path = "/content/gdrive/MyDrive"

### Hardware agnostic settings
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == 'cuda':
    print('Default tensor type is now cuda')
    torch.set_default_tensor_type('torch.cuda.FloatTensor')
print("Device in use is: ", device)

# set random seed to reproduce the work - ensures data is loaded in the same way, but initialisations are still random (not seeded).
# Training is deterministic after model instantiated, so results should be reproducible using r_seed and loading initialisation of network.
r_seed=int(torch.empty((), dtype=torch.int32).random_().item())
torch.manual_seed(r_seed)

# task related settings
task = "scifar10" # will accept scifar10, psmnist, or smnist

if task == "scifar10":
    input_size = 3 # rgb
    data_dir = gdrive_path+"/Users/carlrichardson/Documents/Python/LurieNet/Data/CIFAR10/"
else:
    input_size = 1 # black and white
    data_dir = gdrive_path+"/Users/carlrichardson/Documents/Python/LurieNet/Data/"
output_size = 10 # same for all tasks

# data
train_loader, val_loader, test_loader = getData(task, data_dir, device, r_seed)

# Model settings
n=256 # dimension of state
m=256 # dimension of input to nonlinearity
A=torch.eye(n,n) # can optionally pass A matrix as a fixed parameter
B=torch.eye(n,m) # can optionally pass B matrix as a fixed parameter
C=torch.eye(m,n) # can optionally pass C matrix as a fixed parameter
eta = 0.01 # upper limit on gradient
delta=0.005 # discrete time step of ode
k=1 # dimension of k-compound system
g=1 # slope restriction of nonlinearity
gb=(1/3)*2*(k**2)*eta/delta # max. sum of singular values for B matrix
gc=k # max. sum of singular values for C matrix
ga1=(1/6)*2*(k**2)*eta/delta # bounds eigenvalues of symmetric component of A
ga2=(3/6)*2*(k**2)*eta/delta # bounds terms in skew-symmetric component of the special orthogonal transformation
v1_ind=True # indicates if input v1 is used by model
v2_ind=False # indicates if input v2 is used by model
init = 'He' # specified weight initialisation. Accepts 'He', 'Xavier', 'RNNsofRNNs', 'He+RNNsofRNNs'
model = LurieNet(input_size, output_size, n, m, k, g, gb, gc, ga1, ga2, delta,
                 v1_ind, v2_ind, init, C=C).to(device) # move model to gpu if available

# Training settings.
# Same as RNNs of RNNs paper, however different settings are used after hyperparameter optimization
num_epochs = 100
lr = 1e-3
lr_scale_epochs = [90] # epochs to perform a learning rate cut
lr_scalar = 0.1 # scalar to multiply the learning rate by at the specified epochs
weight_decay = 1e-5

# Optimizer
optim_params = list(model.parameters())
optimizer = torch.optim.Adam(optim_params, lr=lr, weight_decay=weight_decay)

# Loss
criterion = nn.CrossEntropyLoss()

# Datalogging
model_name="RNNsofRNNs"
hyperparam_settings = "k%d_lr%.3f_delta%.3f_1ga%.3f_2ga%.3f_gb%.3f_gc%.3f" % (k, lr, delta, ga1, ga2, gb, gc)
model_dir= os.path.join(gdrive_path+"/Users/carlrichardson/Documents/Python/LurieNet/Models/"+model_name+"/"+task,hyperparam_settings)
log_dir= os.path.join(gdrive_path+"/Users/carlrichardson/Documents/Python/LurieNet/Experiments/"+model_name+"/"+task,hyperparam_settings)
if not os.path.isdir(model_dir):
    os.mkdir(model_dir)
else:
     raise ValueError(f"Do not overwrite existing data!")
if not os.path.isdir(log_dir):
    os.mkdir(log_dir)
else:
     raise ValueError(f"Do not overwrite existing data!")

# save initialization of network
torch.save(model.state_dict(), os.path.join(model_dir,"epoch0.pt"))

# dictionaries for logging all information about experiment and hyperparameters
exp_setup = {'r_seed': r_seed, 'device': device,  'model_name': model_name, 'model_dir': model_dir, \
'log_dir': log_dir, 'task':task}
model_hyperparameters = {'n':n, 'm':m, 'eta':eta, 'k':k, 'g':g, 'gb':gb, 'gc':gc, 'ga1':ga1, 'ga2':ga2, 'delta':delta, \
                   'v1_ind':v1_ind, 'v2_ind':v2_ind, 'init':init, 'A':A, 'B':B, 'C':C}
train_hyperparameters = {'num_epochs':num_epochs, 'lr':lr, 'lr_scale_epochs':lr_scale_epochs, \
                         'lr_scalar':lr_scalar, 'weight_decay':weight_decay, 'optimizer':optimizer, \
                            'criterion': criterion}

# logging experiments and hyperparameters
with open(log_dir+'/exp_setup.pkl', 'wb') as f:
    pickle.dump(exp_setup, f)
    f.close()
with open(log_dir+'/model_hyperparameters.pkl', 'wb') as f:
    pickle.dump(model_hyperparameters, f)
    f.close()
with open(log_dir+'/train_hyperparameters.pkl', 'wb') as f:
    pickle.dump(train_hyperparameters, f)
    f.close()

print('Total params. being trained: %.3fk \n' % (sum(p.numel() for p in model.parameters())/1000.0))
if A!=None or B!=None or C!=None:
    print("Have all hyperparameters been correctly set? Be particularly careful when passing in A, B or C!\n")
else:
    print("Have all hyperparameters been correctly set? \n")

# train
model, optimizer, test_accs, train_losses = train(task, model, optimizer, criterion, train_loader, test_loader, num_epochs,
                                                    lr_scalar, lr_scale_epochs, device, log_dir, model_dir)

Drive already mounted at /content/gdrive; to attempt to forcibly remount, call drive.mount("/content/gdrive", force_remount=True).
Default tensor type is now cuda
Device in use is:  cuda
Files already downloaded and verified
Files already downloaded and verified
Total params. being trained: 134.921k 

Have all hyperparameters been correctly set? Be particularly careful when passing in A, B or C!



  0%|          | 0/100 [00:00<?, ?it/s]

Epoch 1, mean batch loss 1.940137885093689


  1%|          | 1/100 [08:33<14:07:09, 513.42s/it]

Accuracy of the network on the 10000 test images: 35 %
Epoch 2, mean batch loss 1.857492769877116


...

Accuracy of the network on the 10000 test images: 40 %
Epoch 21, mean batch loss 1.692144641717275


 21%|██        | 21/100 [2:58:09<11:08:45, 507.92s/it]

Accuracy of the network on the 10000 test images: 40 %


 21%|██        | 21/100 [3:02:48<11:27:43, 522.32s/it]


KeyboardInterrupt: 

In [ ]:
# '''
# 2. Colab: resume training run. Just need to specify hyperparam_settings, log_dir (has same meaning as above) and
# latest_model from model_dir.
# '''

# # settings for running training loop using a specified Google Drive account
# from google.colab import drive
# drive.mount('/content/gdrive')
# gdrive_path = "/content/gdrive/MyDrive"

# # User input settings
# eta = 0.01 # upper limit on gradient
# delta=0.005 # discrete time step of ode
# k=1 # dimension of k-compound system
# g=1 # slope restriction of nonlinearity
# gb=(1/3)*2*(k**2)*eta/delta # max. sum of singular values for B matrix
# gc=k # max. sum of singular values for C matrix
# ga1=(1/3)*2*(k**2)*eta/delta # bounds eigenvalues of symmetric component of A
# ga2=(1/3)*2*(k**2)*eta/delta
# lr = 0.01

# model_name = "RNNsofRNNs"
# task = "scifar10"
# hyperparam_settings = "k%d_lr%.3f_delta%.3f_1ga%.3f_2ga%.3f_gb%.3f_gc%.3f" % (k, lr, delta, ga1, ga2, gb, gc)
# log_dir= os.path.join(gdrive_path+"/Users/carlrichardson/Documents/Python/LurieNet/Experiments/"+model_name+"/"+task, hyperparam_settings)
# latest_model = "epoch9.pt"

# ### Hardware agnostic settings
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# if device.type == 'cuda':
#     print('Default tensor type is now cuda')
#     torch.set_default_tensor_type('torch.cuda.FloatTensor')
# print("Device in use is: ", device)

# # load dictionaries containing experimental setup, model hyperparameters, and training hyperparameters from log_dir.
# with open(log_dir+'/exp_setup.pkl', 'rb') as f:
#     exp_setup = pickle.load(f)
# with open(log_dir+'/model_hyperparameters.pkl', 'rb') as f:
#     model_hyperparameters = pickle.load(f)
# with open(log_dir+'/train_hyperparameters.pkl', 'rb') as f:
#     train_hyperparameters = pickle.load(f)

# # unpack experimental setup
# r_seed=exp_setup['r_seed']
# torch.manual_seed(r_seed)
# task = exp_setup['task']
# if task == "scifar10":
#     input_size = 3 # rgb
#     data_dir = gdrive_path+"/Users/carlrichardson/Documents/Python/LurieNet/Data/CIFAR10/"
# else:
#     input_size = 1 # black and white
#     data_dir = gdrive_path+"/Users/carlrichardson/Documents/Python/LurieNet/Data/"
# output_size = 10 # same for all tasks
# # device = exp_setup['device']
# model_name=exp_setup['model_name']
# model_dir=exp_setup['model_dir']

# # reload data
# train_loader, val_loader, test_loader = getData(task, data_dir, device, r_seed)

# # unpack model hyperparameters.
# n=model_hyperparameters['n']
# m=model_hyperparameters['m']
# eta=model_hyperparameters['eta']
# k=model_hyperparameters['k']
# g=model_hyperparameters['g']
# gb=model_hyperparameters['gb']
# gc=model_hyperparameters['gc']
# ga1=model_hyperparameters['ga1']
# ga2=model_hyperparameters['ga2']
# delta=model_hyperparameters['delta']
# v1_ind=model_hyperparameters['v1_ind']
# v2_ind=model_hyperparameters['v2_ind']
# A=model_hyperparameters['A']
# B=model_hyperparameters['B']
# C=model_hyperparameters['C']
# init=model_hyperparameters['init']

# # unpack training hyperparameters
# num_epochs = train_hyperparameters['num_epochs']
# lr = train_hyperparameters['lr']
# lr_scale_epochs = train_hyperparameters['lr_scale_epochs']
# lr_scalar = train_hyperparameters['lr_scalar']
# weight_decay = train_hyperparameters['weight_decay']

# # Loss
# criterion = train_hyperparameters['criterion']

# # # load model
# model = LurieNet(input_size, output_size, n, m, k, g, gb, gc, ga1, ga2, delta,
#                  v1_ind, v2_ind, init, C=C).to(device) # move model to gpu if available
# model.load_state_dict(torch.load(os.path.join(model_dir,latest_model)))

# # Optimizer
# optim_params = list(model.parameters())
# optimizer = torch.optim.Adam(optim_params, lr=lr, weight_decay=weight_decay)

# # Loading stats.
# df = pd.read_csv(os.path.join(log_dir, "stats.csv"))
# test_accs = df['test_acc'].tolist()
# train_losses = df['loss'].tolist()
# epochs_list = df['epoch'].tolist()
# min_epoch = epochs_list[-1] # resume on this number since epoch_log was adding 1 to the epoch each time.
# if model.A==None:
#     SA1_max = df['SA1_max'].tolist()
#     SA1_min = df['SA1_min'].tolist()
#     SA2_max = df['SA2_max'].tolist()
#     SA2_min = df['SA2_min'].tolist()
# else:
#     SA1_max = []
#     SA1_min = []
#     SA2_max = []
#     SA2_min = []
# if model.B==None:
#     SB_max = df['SB_max'].tolist()
#     SB_min = df['SB_min'].tolist()
# else:
#     SB_max = []
#     SB_min = []
# if model.C==None:
#     SC_max = df['SC_max'].tolist()
#     SC_min = df['SC_min'].tolist()
# else:
#     SC_max = []
#     SC_min = []

# print('Total params. being trained: %.3fk \n' % (sum(p.numel() for p in model.parameters())/1000.0))

# # # # train
# model, optimizer, test_accs, train_losses = train(task, model, optimizer, criterion, train_loader, test_loader, num_epochs, lr_scalar, lr_scale_epochs, device,
#                                                     log_dir, model_dir, min_epoch=min_epoch, test_accs=test_accs, train_losses=train_losses, epochs_list=epochs_list,
#                                                   SA1_max=SA1_max, SA1_min=SA1_min, SA2_max=SA2_max, SA2_min=SA2_min, SB_max=SB_max, SB_min=SB_min, SC_max=SC_max, SC_min=SC_min)

In [ ]:
# '''
# 3. Cpu: new training run - use this cell when initialising a new training run on cpu.
# Use cell below to resume a training run.
# '''

# ### Hardware agnostic settings
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# if device.type == 'cuda':
#     print('Default tensor type is now cuda')
#     torch.set_default_tensor_type('torch.cuda.FloatTensor')
# print("Device in use is: ", device)

# # set random seed to reproduce the work - ensures data is loaded in the same way, but initialisations are still random (not seeded).
# # Training is deterministic after model instantiated, so results should be reproducible using r_seed and loading initialisation of network.
# r_seed=int(torch.empty((), dtype=torch.int32).random_().item())
# torch.manual_seed(r_seed)

# # task related settings
# task = "scifar10" # will accept scifar10, psmnist, or smnist

# if task == "scifar10":
#     input_size = 3 # rgb
#     data_dir = "/Users/carlrichardson/Documents/Python/LurieNet/Data/CIFAR10/"
# else:
#     input_size = 1 # black and white
#     data_dir = "/Users/carlrichardson/Documents/Python/LurieNet/Data/"
# output_size = 10 # same for all tasks

# # data
# train_loader, val_loader, test_loader = getData(task, data_dir, device, r_seed)

# # Model settings
# n=256 # dimension of state
# m=256 # dimension of input to nonlinearity
# A=torch.eye(n,n) # can optionally pass A matrix as a fixed parameter
# B=torch.eye(n,m) # can optionally pass B matrix as a fixed parameter
# C=torch.eye(m,n) # can optionally pass C matrix as a fixed parameter
# eta = 0.01 # upper limit on gradient
# delta=0.002 # discrete time step of ode
# k=1 # dimension of k-compound system
# g=1 # slope restriction of nonlinearity
# gb=2*(k**2)*eta/(3*delta) # max. sum of singular values for B matrix
# gc=k # max. sum of singular values for C matrix
# ga1=2*(k**2)*eta/(3*delta) # bounds eigenvalues of symmetric component of A
# ga2=2*(k**2)*eta/(3*delta) # bounds terms in skew-symmetric component of the special orthogonal transformation
# v1_ind=True # indicates if input v1 is used by model
# v2_ind=False # indicates if input v2 is used by model
# init = 'He' # specified weight initialisation. Accepts 'He', 'Xavier', 'RNNsofRNNs', 'He+RNNsofRNNs'
# model = LurieNet(input_size, output_size, n, m, k, g, gb, gc, ga1, ga2, delta,
#                  v1_ind, v2_ind, init, C=C).to(device) # move model to gpu if available

# # Training settings.
# # Same as RNNs of RNNs paper, however different settings are used after hyperparameter optimization
# num_epochs = 100
# lr = 0.001
# lr_scale_epochs = [90] # epochs to perform a learning rate cut
# lr_scalar = 0.1 # scalar to multiply the learning rate by at the specified epochs
# weight_decay = 1e-5

# # Optimizer
# optim_params = list(model.parameters())
# optimizer = torch.optim.Adam(optim_params, lr=lr, weight_decay=weight_decay)

# # Loss
# criterion = nn.CrossEntropyLoss()

# # Datalogging
# model_name="RNNsofRNNs"
# hyperparam_settings = "k%d_lr%.3f_delta%.3f_1ga%.3f_2ga%.3f_gb%.3f_gc%.3f" % (k, lr, delta, ga1, ga2, gb, gc)
# model_dir= os.path.join("/Users/carlrichardson/Documents/Python/LurieNet/Models/"+model_name+"/"+task,hyperparam_settings)
# log_dir= os.path.join("/Users/carlrichardson/Documents/Python/LurieNet/Experiments/"+model_name+"/"+task,hyperparam_settings)
# if not os.path.isdir(model_dir):
#     os.mkdir(model_dir)
# else:
#      raise ValueError(f"Do not overwrite existing data!")
# if not os.path.isdir(log_dir):
#     os.mkdir(log_dir)
# else:
#      raise ValueError(f"Do not overwrite existing data!")

# # save initialization of network
# torch.save(model.state_dict(), os.path.join(model_dir,"epoch0.pt"))

# # dictionaries for logging all information about experiment and hyperparameters
# exp_setup = {'r_seed': r_seed, 'device': device,  'model_name': model_name, 'model_dir': model_dir, \
# 'log_dir': log_dir, 'task':task}
# model_hyperparameters = {'n':n, 'm':m, 'eta':eta, 'k':k, 'g':g, 'gb':gb, 'gc':gc, 'ga1':ga1, 'ga2':ga2, 'delta':delta, \
#                    'v1_ind':v1_ind, 'v2_ind':v2_ind, 'init':init, 'A':A, 'B':B, 'C':C}
# train_hyperparameters = {'num_epochs':num_epochs, 'lr':lr, 'lr_scale_epochs':lr_scale_epochs, \
#                          'lr_scalar':lr_scalar, 'weight_decay':weight_decay, 'optimizer':optimizer, \
#                             'criterion': criterion}

# # logging experiments and hyperparameters
# with open(log_dir+'/exp_setup.pkl', 'wb') as f:
#     pickle.dump(exp_setup, f)
#     f.close()
# with open(log_dir+'/model_hyperparameters.pkl', 'wb') as f:
#     pickle.dump(model_hyperparameters, f)
#     f.close()
# with open(log_dir+'/train_hyperparameters.pkl', 'wb') as f:
#     pickle.dump(train_hyperparameters, f)
#     f.close()

# print('Total params. being trained: %.3fk \n' % (sum(p.numel() for p in model.parameters())/1000.0))
# if A!=None or B!=None or C!=None:
#     print("Have all hyperparameters been correctly set? Be particularly careful when passing in A, B or C!\n")
# else:
#     print("Have all hyperparameters been correctly set? \n")

# # train
# model, optimizer, test_accs, train_losses = train(task, model, optimizer, criterion, train_loader, test_loader, num_epochs,
#                                                     lr_scalar, lr_scale_epochs, device, log_dir, model_dir)


In [ ]:
# '''
# 4. Cpu: resume training run. Just need to specify log_dir (has same meaning as above) and
# latest_model from model_dir.
# '''

# # User input settings
# # k =
# # lr =
# # delta =
# # ga1 =
# # ga2 =
# # gb =
# # gc =
# model_name = "RNNsofRNNs"
# task = "scifar10"
# hyperparam_settings = "k%d_lr%.3f_delta%.3f_1ga%.3f_2ga%.3f_gb%.3f_gc%.3f" % (k, lr, delta, ga1, ga2, gb, gc)
# log_dir= os.path.join("/Users/carlrichardson/Documents/Python/LurieNet/Experiments/"+model_name+"/"+task, hyperparam_settings)
# latest_model = "epoch1.pt"

# ### Hardware agnostic settings
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# if device.type == 'cuda':
#     print('Default tensor type is now cuda')
#     torch.set_default_tensor_type('torch.cuda.FloatTensor')
# print("Device in use is: ", device)

# # load dictionaries containing experimental setup, model hyperparameters, and training hyperparameters from log_dir.
# with open(log_dir+'/exp_setup.pkl', 'rb') as f:
#     exp_setup = pickle.load(f)
# with open(log_dir+'/model_hyperparameters.pkl', 'rb') as f:
#     model_hyperparameters = pickle.load(f)
# with open(log_dir+'/train_hyperparameters.pkl', 'rb') as f:
#     train_hyperparameters = pickle.load(f)

# # unpack experimental setup
# r_seed=exp_setup['r_seed']
# torch.manual_seed(r_seed)
# task = exp_setup['task']
# if task == "scifar10":
#     input_size = 3 # rgb
#     data_dir = "/Users/carlrichardson/Documents/Python/LurieNet/Data/CIFAR10/"
# else:
#     input_size = 1 # black and white
#     data_dir = "/Users/carlrichardson/Documents/Python/LurieNet/Data/"
# output_size = 10 # same for all tasks
# # device = exp_setup['device']
# model_name=exp_setup['model_name']
# model_dir=exp_setup['model_dir']

# # reload data
# train_loader, val_loader, test_loader = getData(task, data_dir, device, r_seed)

# # unpack model hyperparameters.
# n=model_hyperparameters['n']
# m=model_hyperparameters['m']
# eta=model_hyperparameters['eta']
# k=model_hyperparameters['k']
# g=model_hyperparameters['g']
# gb=model_hyperparameters['gb']
# gc=model_hyperparameters['gc']
# ga1=model_hyperparameters['ga1']
# ga2=model_hyperparameters['ga2']
# delta=model_hyperparameters['delta']
# v1_ind=model_hyperparameters['v1_ind']
# v2_ind=model_hyperparameters['v2_ind']
# A=model_hyperparameters['A']
# B=model_hyperparameters['B']
# C=model_hyperparameters['C']
# init=model_hyperparameters['init']

# # unpack training hyperparameters
# num_epochs = 3 # train_hyperparameters['num_epochs']
# lr = train_hyperparameters['lr']
# lr_scale_epochs = train_hyperparameters['lr_scale_epochs']
# lr_scalar = train_hyperparameters['lr_scalar']
# weight_decay = train_hyperparameters['weight_decay']

# # Loss
# criterion = train_hyperparameters['criterion']

# # # load model
# model = LurieNet(input_size, output_size, n, m, k, g, gb, gc, ga1, ga2, delta,
#                  v1_ind, v2_ind, init, C=C).to(device) # move model to gpu if available
# model.load_state_dict(torch.load(os.path.join(model_dir,latest_model)))

# # Optimizer
# optim_params = list(model.parameters())
# optimizer = torch.optim.Adam(optim_params, lr=lr, weight_decay=weight_decay)

# # Loading stats.
# df = pd.read_csv(os.path.join(log_dir, "stats.csv"))
# test_accs = df['test-acc'].tolist()
# train_losses = df['loss'].tolist()
# epochs_list = df['epoch'].tolist()
# min_epoch = epochs_list[-1] # resume on this number since epoch_log was adding 1 to the epoch each time.
# if model.A==None:
#     SA1_max = df['SA1_max'].tolist()
#     SA1_min = df['SA1_min'].tolist()
#     SA2_max = df['SA2_max'].tolist()
#     SA2_min = df['SA2_min'].tolist()
# else:
#     SA1_max = []
#     SA1_min = []
#     SA2_max = []
#     SA2_min = []
# if model.B==None:
#     SB_max = df['SB_max'].tolist()
#     SB_min = df['SB_min'].tolist()
# else:
#     SB_max = []
#     SB_min = []
# if model.C==None:
#     SC_max = df['SC_max'].tolist()
#     SC_min = df['SC_min'].tolist()
# else:
#     SC_max = []
#     SC_min = []

# print('Total params. being trained: %.3fk \n' % (sum(p.numel() for p in model.parameters())/1000.0))

# # # # train
# model, optimizer, test_accs, train_losses = train(task, model, optimizer, criterion, train_loader, test_loader, num_epochs, lr_scalar, lr_scale_epochs, device,
#                                                     log_dir, model_dir, min_epoch=min_epoch, test_accs=test_accs, train_losses=train_losses, epochs_list=epochs_list,
#                                                   SA1_max=SA1_max, SA1_min=SA1_min, SA2_max=SA2_max, SA2_min=SA2_min, SB_max=SB_max, SB_min=SB_min, SC_max=SC_max, SC_min=SC_min)

    Test scripts for loading, preprocessing and splitting data section.

In [ ]:
# Test scripts for getData()

# Test 1 - scifar10 - verified shape and data type of batches for each data loader.
# task='scifar10'
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# dir = "/Users/carlrichardson/Documents/Python/LurieNet/Data/CIFAR10/" # when task = "scifar10"
# train_loader, val_loader, test_loader = getData(task,dir,device)

# train_features, train_labels = next(iter(train_loader))
# print(f"Feature batch shape: {train_features.size()}")
# print(f"Labels batch shape: {train_labels.size()}")
# print(f"Feature batch type: {train_features.type()}")
# print(f"Labels batch type: {train_labels.type()}")
# assert train_features.size() == torch.Size([64, 3, 32, 32]) # i.e., [batch size, rgb, width, height]
# assert train_labels.size() == torch.Size([64]) # i.e., [batch size]
# assert train_features.type() == 'torch.FloatTensor' or train_features.type() == 'torch.cuda.FloatTensor'
# assert train_labels.type() == 'torch.LongTensor' or train_labels.type() == 'torch.cuda.LongTensor'

# val_features, val_labels = next(iter(val_loader))
# print(f"Feature batch shape: {val_features.size()}")
# print(f"Labels batch shape: {val_labels.size()}")
# print(f"Feature batch type: {val_features.type()}")
# print(f"Labels batch type: {val_labels.type()}")
# assert val_features.size() == torch.Size([1024, 3, 32, 32]) # i.e., [batch size, rgb, width, height]
# assert val_labels.size() == torch.Size([1024]) # i.e., [batch size]
# assert val_features.type() == 'torch.FloatTensor' or val_features.type() == 'torch.cuda.FloatTensor'
# assert val_labels.type() == 'torch.LongTensor' or val_labels.type() == 'torch.cuda.LongTensor'

# test_features, test_labels = next(iter(test_loader))
# print(f"Feature batch shape: {test_features.size()}")
# print(f"Labels batch shape: {test_labels.size()}")
# print(f"Feature batch type: {test_features.type()}")
# print(f"Labels batch type: {test_labels.type()}")
# assert test_features.size() == torch.Size([1024, 3, 32, 32]) # i.e., [batch size, rgb, width, height]
# assert test_labels.size() == torch.Size([1024]) # i.e., [batch size]
# assert test_features.type() == 'torch.FloatTensor' or test_features.type() == 'torch.cuda.FloatTensor'
# assert test_labels.type() == 'torch.LongTensor' or test_labels.type() == 'torch.cuda.LongTensor'

# # Test 2 - smnist - verified shape and data type of batches for each data loader.
# task='smnist'
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# dir = "/Users/carlrichardson/Documents/Python/LurieNet/Data/" # when task = "smnist" = "psmnist"
# train_loader, val_loader, test_loader = getData(task,dir,device)

# train_features, train_labels = next(iter(train_loader))
# print(f"Feature batch shape: {train_features.size()}")
# print(f"Labels batch shape: {train_labels.size()}")
# print(f"Feature batch type: {train_features.type()}")
# print(f"Labels batch type: {train_labels.type()}")
# assert train_features.size() == torch.Size([64, 1, 28, 28]) # i.e., [batch size, grayscale, width, height]
# assert train_labels.size() == torch.Size([64]) # i.e., [batch size]
# assert train_features.type() == 'torch.FloatTensor' or train_features.type() == 'torch.cuda.FloatTensor'
# assert train_labels.type() == 'torch.LongTensor' or train_labels.type() == 'torch.cuda.LongTensor'

# val_features, val_labels = next(iter(val_loader))
# print(f"Feature batch shape: {val_features.size()}")
# print(f"Labels batch shape: {val_labels.size()}")
# print(f"Feature batch type: {val_features.type()}")
# print(f"Labels batch type: {val_labels.type()}")
# assert val_features.size() == torch.Size([1024, 1, 28, 28]) # i.e., [batch size, grayscale, width, height]
# assert val_labels.size() == torch.Size([1024]) # i.e., [batch size]
# assert val_features.type() == 'torch.FloatTensor' or val_features.type() == 'torch.cuda.FloatTensor'
# assert val_labels.type() == 'torch.LongTensor' or val_labels.type() == 'torch.cuda.LongTensor'

# test_features, test_labels = next(iter(test_loader))
# print(f"Feature batch shape: {test_features.size()}")
# print(f"Labels batch shape: {test_labels.size()}")
# print(f"Feature batch type: {test_features.type()}")
# print(f"Labels batch type: {test_labels.type()}")
# assert test_features.size() == torch.Size([1024, 1, 28, 28]) # i.e., [batch size, grayscale, width, height]
# assert test_labels.size() == torch.Size([1024]) # i.e., [batch size]
# assert test_features.type() == 'torch.FloatTensor' or test_features.type() == 'torch.cuda.FloatTensor'
# assert test_labels.type() == 'torch.LongTensor' or test_labels.type() == 'torch.cuda.LongTensor'

# # Test 3 - psmnist - verified shape and data type of batches for each data loader.
# task='psmnist'
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# dir = "/Users/carlrichardson/Documents/Python/LurieNet/Data/" # when task = "smnist" = "psmnist"
# train_loader, val_loader, test_loader = getData(task,dir,device)

# train_features, train_labels = next(iter(train_loader))
# print(f"Feature batch shape: {train_features.size()}")
# print(f"Labels batch shape: {train_labels.size()}")
# print(f"Feature batch type: {train_features.type()}")
# print(f"Labels batch type: {train_labels.type()}")
# assert train_features.size() == torch.Size([64, 1, 28, 28]) # i.e., [batch size, grayscale, width, height]
# assert train_labels.size() == torch.Size([64]) # i.e., [batch size]
# assert train_features.type() == 'torch.FloatTensor' or train_features.type() == 'torch.cuda.FloatTensor'
# assert train_labels.type() == 'torch.LongTensor' or train_labels.type() == 'torch.cuda.LongTensor'

# val_features, val_labels = next(iter(val_loader))
# print(f"Feature batch shape: {val_features.size()}")
# print(f"Labels batch shape: {val_labels.size()}")
# print(f"Feature batch type: {val_features.type()}")
# print(f"Labels batch type: {val_labels.type()}")
# assert val_features.size() == torch.Size([1024, 1, 28, 28]) # i.e., [batch size, grayscale, width, height]
# assert val_labels.size() == torch.Size([1024]) # i.e., [batch size]
# assert val_features.type() == 'torch.FloatTensor' or val_features.type() == 'torch.cuda.FloatTensor'
# assert val_labels.type() == 'torch.LongTensor' or val_labels.type() == 'torch.cuda.LongTensor'

# test_features, test_labels = next(iter(test_loader))
# print(f"Feature batch shape: {test_features.size()}")
# print(f"Labels batch shape: {test_labels.size()}")
# print(f"Feature batch type: {test_features.type()}")
# print(f"Labels batch type: {test_labels.type()}")
# assert test_features.size() == torch.Size([1024, 1, 28, 28]) # i.e., [batch size, grayscale, width, height]
# assert test_labels.size() == torch.Size([1024]) # i.e., [batch size]
# assert test_features.type() == 'torch.FloatTensor' or test_features.type() == 'torch.cuda.FloatTensor'
# assert test_labels.type() == 'torch.LongTensor' or test_labels.type() == 'torch.cuda.LongTensor'

    Test scripts for defining the model section.

In [ ]:
#### Model tests

# # Model settings
# device = torch.device("cpu")
# input_size=3
# output_size=5
# n=2 # dimension of state **
# m=2 # dimension of input to nonlinearity **
# A=torch.eye(n,n) # can optionally pass A matrix as a fixed parameter
# B=torch.eye(n,m) # can optionally pass B matrix as a fixed parameter
# C=torch.eye(m,n) # can optionally pass C matrix as a fixed parameter
# k=2 # dimension of k-compound system
# g=1 # slope restriction of nonlinearity
# gb=3 # max. sum of singular values for B matrix **
# gc=3 # max. sum of singular values for C matrix ** THIS VALUE IS BASED ON C=I
# ga1=3 # bounds eigenvalues of symmetric component of A when k=1 **
# ga2=3 # bounds terms in skew-symmetric component of the special orthogonal transformation **
# delta=0.03 # discrete time step of ode
# v1_ind=True # indicates if input v1 is used by model
# v2_ind=True # indicates if input v2 is used by model
# init = 'RNNsofRNNs' # specified weight initialisation. Accepts 'He', 'Xavier', 'RNNsofRNNs', 'He+RNNsofRNNs'
# test_model = LurieNet(input_size, output_size, n, m, k, g, gb, gc, ga1, ga2, delta, v1_ind, v2_ind, init).to(device) # move model to gpu if available

# test_input = torch.normal(0,1,(4, 2, 3)) # shape [batch_size, pixels/image, rgb=input_size]
# test_out, test_dict = test_model(test_input)

## Test 1 - Are C components parametrized correctly?
# print(torch.inverse(test_dict['UC']))
# print(test_dict['UC'].t())
# print(torch.inverse(test_dict['VC']))
# print(test_dict['VC'].t())
# assert torch.allclose(torch.inverse(test_dict['UC']), test_dict['UC'].t()) # will raise error if weight isn't orthogonal - tempremental due to numerical issues?
# assert torch.allclose(torch.inverse(test_dict['VC']), test_dict['VC'].t()) # will raise error if weight isn't orthogonal
# assert torch.max(test_dict['SC']) <= gc/k # will raise error if max weight is greater than gc/k

# ## Test 2 - Are B components parametrized correctly?
# print(torch.inverse(test_dict['UB']))
# print(test_dict['UB'].t())
# print(torch.inverse(test_dict['VB']))
# print(test_dict['VB'].t())
# assert torch.allclose(torch.inverse(test_dict['UB']), test_dict['UB'].t()) # will raise error if weight isn't orthogonal
# assert torch.allclose(torch.inverse(test_dict['VB']), test_dict['VB'].t()) # will raise error if weight isn't orthogonal
# assert torch.max(test_dict['SB']) <= gb/k # will raise error if max weight is greater than gb/k

# ## Test 3 - Are A components parametrized correctly?
# print(torch.inverse(test_dict['UA1']))
# print(test_dict['UA1'].t())
# print(torch.inverse(test_dict['UA2']))
# print(test_dict['UA2'].t())
# assert torch.allclose(torch.inverse(test_dict['UA1']), test_dict['UA1'].t()) # will raise error if weight isn't orthogonal
# assert torch.allclose(torch.inverse(test_dict['UA2']), test_dict['UA2'].t()) # will raise error if weight isn't orthogonal
# assert torch.max(test_dict['SA2']) <= ga2 # will raise error if max weight is greater than ga2
# assert torch.all(test_dict['SA2'].eq(-test_dict['SA2'].t())) # will raise error if input isnt skew-symmetric
# if k==1:
#     td = torch.diag(test_dict['SA1'])
#     td = td.sort(descending=True)
#     td = td[0]
#     print(td)
#     print(td[0])
#     print("td max: ", -2*g*gb*gc)
#     print(td[n-1])
#     print("td min: ", -2*g*gb*gc - ga1)
# else:
#     td = torch.diag(test_dict['SA1'])
#     td = td.sort(descending=True)
#     td = td[0]
#     sk = torch.sum(td[0:k])
#     sk_ = torch.sum(td[0:k-1])
#     print(td)
#     print(sk)
#     print(sk_)
#     print("sk max: ", -2*g*gb*gc/k)
#     print("sk_ min: ", -2*g*gb*gc/(k-1))
#     assert sk < -2*g*gb*gc/k and sk_ >= -2*g*gb*gc/(k-1) # will raise error if max and min are outside expected range

# # Test 4 - Simulate system and check dynamics are k-contracting

# A = 0.5*md([test_dict['UA1'], test_dict['SA1'], torch.transpose(test_dict['UA1'],0,1)]) + \
#     0.5*md([test_dict['UA2'], test_dict['SA2'], torch.transpose(test_dict['UA2'],0,1)])
# B = md([test_dict['UB'], test_dict['SB'], torch.transpose(test_dict['VB'],0,1)])
# C = md([test_dict['UC'], test_dict['SC'], torch.transpose(test_dict['VC'],0,1)])
# N = 100
# T = 100
# X0 = torch.normal(0,2, (N, n)) # shape [batch_size, n]
# X = torch.zeros(N,T,n) # [batch size, t, n]
# X[:,0,:] = X0
# Y = torch.zeros(N,T,m)
# V1 = torch.ones(N,T,n)
# V2 = torch.ones(N,T,m)
# # # Loop through pixels in an image. inputs[:,i,:] has shape [batch_size, input_size] and inputs has shape [batch_size, pixels, input_size]
# for i in range(0,T-1):
#     Y[:,i,:] = torch.matmul(X[:,i,:], torch.transpose(C,0,1)) + V2[:,i,:] # Y has shape [batch size, m]
#     X[:,i+1,:] = X[:,i,:] + delta*(torch.matmul(X[:,i,:], torch.transpose(A,0,1)) + torch.matmul(F.relu(Y[:,i,:]), torch.transpose(B,0,1)) + V1[:,i,:]) # X has shape [batch size, n]

# # scatter plots of X[:,0,:] and X[:,T,:]
# X = X.detach().numpy()
# plt.scatter(X[:,0,0], X[:,0,1], c='b')
# plt.scatter(X[:,T-1,0], X[:,T-1,1], c='r')